Hits-per-ACR Distribution

In [ ]:
import pandas as pd
from Bio import SeqIO


def analyze_acr_motif_distribution():

    # --------------------------------------------------
    # 1. Read ACR IDs and FIMO results
    # --------------------------------------------------

    acr_ids = sorted({
        record.id.strip()
        for record in SeqIO.parse(
            "/Users/avery/Downloads/dACR.fa", "fasta"
        )
    })

    gff = pd.read_csv(
       "/Users/avery/fimo_collapsed_with_metadata.csv"
    )

    # Normalize sequence identifiers
    gff["seqid"] = gff["seqid"].astype(str).str.strip()

    # Keep only rows corresponding to ACRs in the FASTA
    gff = gff[gff["seqid"].isin(acr_ids)].copy()

    total_acrs = len(acr_ids)

    # --------------------------------------------------
    # 2. Count retained motif-hit rows per ACR
    # --------------------------------------------------

    # Each row represents one retained motif-hit record
    hit_counts = (
        gff.groupby("seqid")
        .size()
        .reindex(acr_ids, fill_value=0)
    )

    zero_hits = (hit_counts == 0).sum()
    hits_only = hit_counts[hit_counts > 0]

    print("=" * 55)
    print("ACR MOTIF-HIT SUMMARY")
    print("=" * 55)

    print(f"Total ACRs: {total_acrs:,}")
    print(
        f"ACRs with 0 hits: {zero_hits:,} "
        f"({zero_hits / total_acrs * 100:.2f}%)"
    )
    print(
        f"ACRs with hits: {len(hits_only):,} "
        f"({len(hits_only) / total_acrs * 100:.2f}%)"
    )

    if not hits_only.empty:
        print("\nRetained motif-hit rows per ACR (ACRs with hits only):")
        print(f"Minimum:  {hits_only.min():.2f}")
        print(f"Q1:       {hits_only.quantile(0.25):.2f}")
        print(f"Median:   {hits_only.median():.2f}")
        print(f"Q3:       {hits_only.quantile(0.75):.2f}")
        print(f"Maximum:  {hits_only.max():.2f}")
        print(f"Average:  {hits_only.mean():.2f}")

    # --------------------------------------------------
    # 3. Distribution of retained hit rows per ACR
    # --------------------------------------------------

    print("\n" + "=" * 55)
    print("MOTIF-HIT FREQUENCY DISTRIBUTION")
    print("=" * 55)

    frequency = hit_counts.value_counts().sort_index()

    distribution = pd.DataFrame({
        "ACR_Count": frequency,
        "Percent_of_All_ACRs": frequency / total_acrs * 100
    })

    print(
        distribution.to_string(
            formatters={
                "Percent_of_All_ACRs": lambda x: f"{x:.2f}%"
            }
        )
    )

    # --------------------------------------------------
    # 4. Unique motifs merged per retained hit row
    # --------------------------------------------------

    print("\n" + "=" * 55)
    print("UNIQUE MOTIFS MERGED PER RETAINED HIT")
    print("=" * 55)

    gff["Unique_Motif_Count"] = pd.to_numeric(
        gff["Unique_Motif_Count"],
        errors="coerce"
    )

    unique_motif_counts = gff["Unique_Motif_Count"].dropna()

    print(
        f"Retained rows with available counts: "
        f"{len(unique_motif_counts):,}"
    )

    if not unique_motif_counts.empty:
        print("\nSummary across retained rows:")
        print(f"Minimum:  {unique_motif_counts.min():.2f}")
        print(f"Q1:       {unique_motif_counts.quantile(0.25):.2f}")
        print(f"Median:   {unique_motif_counts.median():.2f}")
        print(f"Q3:       {unique_motif_counts.quantile(0.75):.2f}")
        print(f"Maximum:  {unique_motif_counts.max():.2f}")
        print(f"Average:  {unique_motif_counts.mean():.2f}")

        print("\nFrequency distribution:")
        print(
            unique_motif_counts.value_counts()
            .sort_index()
            .to_string()
        )


# Run analysis
analyze_acr_motif_distribution()

ACR MOTIF-HIT SUMMARY
Total ACRs: 31,295
ACRs with 0 hits: 12,262 (39.18%)
ACRs with hits: 19,033 (60.82%)

Retained motif-hit rows per ACR (ACRs with hits only):
Minimum:  1.00
Q1:       1.00
Median:   2.00
Q3:       4.00
Maximum:  23.00
Average:  3.04

MOTIF-HIT FREQUENCY DISTRIBUTION
    ACR_Count Percent_of_All_ACRs
0       12262              39.18%
1        5450              17.41%
2        4787              15.30%
3        3007               9.61%
4        1967               6.29%
5        1381               4.41%
6         864               2.76%
7         555               1.77%
8         373               1.19%
9         235               0.75%
10        143               0.46%
11        102               0.33%
12         57               0.18%
13         35               0.11%
14         36               0.12%
15         18               0.06%
16         10               0.03%
17          4               0.01%
18          2               0.01%
19          6               0.02